In [1]:
# A set of functions for thepartial differential equation creator/derivations
include("operator_terms.jl")
include("print_terms.jl")

In [2]:
function get_indexes(exp_op::Union{Term, Vector{Term}})::Tuple{Vector{String}, Bool, Bool}
    # Get the indexes of the operators and return an array of the indexes
    spin_indices = exp_op.spin_indices
    # get only the unique and sorted indexes into a single joined string
    indexes = string.(sort(unique(spin_indices)))
    spin_op::Bool = length(exp_op.spin_indices) > 0 ? true : false
    cavity_op::Bool = length(exp_op.bosons) > 0 ? true : false
    return indexes, spin_op, cavity_op
end
# test 
exp_op = make_term("zi*xj*yh+-")
display(get_indexes(exp_op))

(["h", "i", "j"], true, true)

In [3]:
##############################################################################################################################
#### Single Qubit ############################################################################################################
##############################################################################################################################

In [4]:
function Trace_of_D_Operator(Op::Union{Term, Vector{Term}}, exp_op::Union{Term, Vector{Term}}, vars::Union{String,Vector{String}}=String[])::Union{Term, Vector{Term}}
    # returns the dissipator operator D[Op] \rho = Op \rho Op^\dagger - 1/2 (Op^\dagger Op \rho + \rho Op^\dagger Op) times the expected operator
    # Tr[exp_op D[Op] \rho] --> use trace property (Tr[AB] = Tr[BA]) to get \rho on the right side
    # Calculate Op^dagger * exp_op * Op - 1/2 (exp_op * Op^\dagger * Op + Op^\dagger * Op * exp_op)
    # Generate the 3 terms and sum them up
    Op_dag = dagger_term(Op)
    Op_dag_X_Op = multiply_terms(Op_dag, Op)

    # 1st Op^dagger * exp_op * Op
    Op_dag_X_exp_op = multiply_terms(Op_dag, exp_op)
    first_term = multiply_terms(Op_dag_X_exp_op, Op)
    # 2nd exp_op * Op^\dagger * Op    and    3rd Op^\dagger * Op * exp_op
    second_term = add_terms(multiply_terms(exp_op, Op_dag_X_Op), multiply_terms(Op_dag_X_Op, exp_op))
    scale_term!(second_term, -0.5)
    # combine terms
    terms = add_terms(first_term, second_term)
    if length(vars) > 0
        terms = add_vars(terms, vars)
    end
    return terms
end
# Test
Op = make_term("+z")
exp_op = make_term("-x")
display(Trace_of_D_Operator(Op, exp_op, "A")) # should be -2*"-x"

2-element Vector{Term}:
 -2A a†a²x 
 -7A/2 ax 

In [9]:
function Trace_of_Hamiltonian_commutator(exp_op::Union{Term, Vector{Term}}, index::String="")::Union{Term, Vector{Term}}
    # consists of delta_s term, g term and drive term (\sqrt{kappa}, beta)
    # Use Tr(exp_op * [H, rho]) = Tr([exp_op, H] * rho) = Tr([exp_op, H0] * rho) + Tr([exp_op, H1] * rho) + ...
    # for the parts H = H0 + H1 + ...
    index_str::String = ""
    if length(index) > 0
        index_str= "_"*index
    end
    z= make_term("z"*index, 1/2)
    delta_s_terms = add_vars(commutator_terms(exp_op, z, -1.0im), raw"\Delta"*index_str)
    a_sigma_plus = multiply_terms(make_term("-"), sigma_plus(index))  # a_- * sig_+
    a_dag_sigma_minus = multiply_terms(make_term("+"), sigma_minus(index))  # a_+ * sig_-
    g_terms = add_vars(commutator_terms(exp_op, add_terms(a_sigma_plus, a_dag_sigma_minus), -1.0im), "g"*index_str)
    brace_term = [make_term("+", 1.0im, raw"\sqrt{\kappa}\beta"), make_term("-", -1.0im, raw"\sqrt{\kappa}\beta^{*}")]
    drive_terms = commutator_terms(exp_op, brace_term, -1.0im)
    terms = add_terms(delta_s_terms, g_terms, drive_terms)
end
# Test
exp_op = make_term("-")
display(Trace_of_Hamiltonian_commutator(exp_op)) 

3-element Vector{Term}:
 -ig/2 x 
 -g/2 y 
 √̅κβ

In [12]:
mutable struct DE_Term              # Differential Equation Term (for a Single Qubit)
    exp_op::Term                        # Expectation value, for which we derived the time derivative
    which_ind::Vector{String}           # Indexes contained in exp_op
    terms::Vector{Term}                 # Terms
    # Inner constructor with default values
    function DE_Term(exp_op::Term, which_ind::Vector{String}; terms::Vector{Term}=Term[])
        new(exp_op, which_ind, terms)
    end
end

function terms2str(x::DE_Term; group_terms::Bool=true, do_sigma::Bool=false, do_latex::Bool=true, do_braket::Bool=true)::String
    exp_op::Term = x.exp_op
    terms::Vector{Term} = x.terms
    # create the string
    eq_str::String = ""
    exp_op_str::String = term2str(exp_op, do_sigma=do_sigma, do_latex=do_latex, do_braket=do_braket)
    terms_str::String = terms2str(terms, group_terms=group_terms, do_sigma=do_sigma, do_latex=do_latex, do_braket=do_braket)
    if do_latex
        eq_str = "\\frac{\\mathrm{d}}{\\mathrm{d}t}" * exp_op_str * "= " * terms_str
    else
        eq_str = "d/dt" * exp_op_str * " = " * terms_str
    end
    return eq_str
end

function Base.show(io::IO, ::MIME"text/plain", x::DE_Term)
    latex_string = terms2str(x, do_latex=false, do_braket=true)
    print(io, latex_string)
end
function Base.show(io::IO, ::MIME"text/latex", x::DE_Term)
    latex_string = terms2str(x, do_latex=true, do_braket=true)
    print(io, latexstring(latex_string))
end

In [22]:
function time_derivative_expectation_of_operator(exp_op::Term, index::String="")::DE_Term
    # Liouvillian = -i [H, \rho] \rho + \gamma D[\sigma_z] \rho + \Gamma D[\sigma_-] + \kappa D[a] \rho
    which_ind = get_indexes(exp_op)
    de_term::DE_Term = DE_Term(exp_op, which_ind[1])
    term_ham = Trace_of_Hamiltonian_commutator(exp_op, index)
    term_gamma = Trace_of_D_Operator(make_term("z"*index), exp_op, raw"\gamma")
    term_Gamma = Trace_of_D_Operator(sigma_minus(index), exp_op, raw"\Gamma")
    term_kappa = Trace_of_D_Operator(make_term("-"), exp_op, raw"\kappa")
    de_term.terms = add_terms(term_ham, term_gamma, term_Gamma, term_kappa)
    return de_term
end
# Test
exp_op = make_term("--")
display(time_derivative_expectation_of_operator(exp_op))

d/dt⟨aa⟩ = 2√̅κβ√̅κβ⟨a⟩ - κκ⟨a²⟩ - g(i⟨ax⟩ + ⟨ay⟩)

In [8]:
##############################################################################################################################
#### Multi Qubit #############################################################################################################
##############################################################################################################################

In [17]:
function get_delta_and_g_str(index::String="", index_delta::Bool=true, index_g::Bool=True)::Tuple{String, String}
    # Get the delta and g strings
    index_str::String = ""
    if length(index) > 0
        index_str = "_"*index
    end
    delta_str = index_delta ? raw"\Delta"*index_str : raw"\Delta"    # add index to delta if index_Delta is true
    g_str = index_g ? "g"*index_str : "g"    # add index to g if index_g is true
    return delta_str, g_str
end
# test
display(get_delta_and_g_str("i", true, true))

("\\Delta_i", "g_i")

In [18]:
function Trace_of_H_i_rho(exp_op::Union{Term, Vector{Term}}, index::String="", index_delta::Bool=true, index_g::Bool=true)::Vector{Term}
    # consists of delta_s term, g term and drive term (\sqrt{kappa}, beta)
    # Use Tr(exp_op * [H, rho]) = Tr([exp_op, H] * rho) = Tr([exp_op, H0] * rho) + Tr([exp_op, H1] * rho) + ...
    # for the parts H = H0 + H1 + ...
    delta_str, g_str = get_delta_and_g_str(index, index_delta, index_g)
    z = make_term("z"*index, 1/2)
    delta_s_terms = add_vars(commutator_terms(exp_op, z, -1.0im), delta_str)
    a_sigma_plus = multiply_terms(make_term("-"), sigma_plus(index))  # a_- * sig_+
    a_dag_sigma_minus = multiply_terms(make_term("+"), sigma_minus(index))  # a_+ * sig_-
    g_terms = add_vars(commutator_terms(exp_op, add_terms(a_sigma_plus, a_dag_sigma_minus), -1.0im), g_str)
    # last term in H does not contain sum
    brace_term = add_terms(make_term("+", 1.0im, raw"\sqrt{\kappa}\beta"), make_term("-", -1.0im, raw"\sqrt{\kappa}\beta^{*}"))
    drive_terms = commutator_terms(exp_op, brace_term, -1.0im)
    return add_terms(delta_s_terms, g_terms, drive_terms)
end
# Test
exp_op = make_term("-")
display(Trace_of_H_i_rho(exp_op))

3-element Vector{Term}:
 -ig/2 x 
 -g/2 y 
 √̅κβ

In [19]:
function Trace_of_L_i_operator(exp_op::Term, index::String="", index_delta::Bool=true, index_g::Bool=true)::Vector{Term}
    delta_str, g_str = get_delta_and_g_str(index, index_delta, index_g)
    z = make_term("z"*index, 1/2)
    delta_s_terms = add_vars(commutator_terms(exp_op, z, -1.0im), delta_str)
    a_sigma_plus = multiply_terms(make_term("-"), sigma_plus(index))  # a_- * sig_+
    a_dag_sigma_minus = multiply_terms(make_term("+"), sigma_minus(index))  # a_+ * sig_-
    g_terms = add_vars(commutator_terms(exp_op, add_terms(a_sigma_plus, a_dag_sigma_minus), -1.0im), g_str)
    term_gamma = Trace_of_D_Operator(make_term("z"*index), exp_op, raw"\gamma")
    term_Gamma = Trace_of_D_Operator(sigma_minus(index), exp_op, raw"\Gamma")
    return add_terms(delta_s_terms, g_terms, term_gamma, term_Gamma)
end
# Test
exp_op = make_term("-xi*yj")
display(Trace_of_L_i_operator(exp_op, "h"))

2-element Vector{Term}:
 -ig_h/2 xₕxᵢyⱼ 
 -g_h/2 yₕxᵢyⱼ 

In [20]:
mutable struct DE_Term_Multi          # Differential Equation Term (for Multi Qubit)
    exp_op::Term                        # Expectation value, for which we derived the time derivative
    which_ind::Vector{String}           # Indexes contained in exp_op
    sum_ind::String                     # Indexes to sum over
    sum_i_terms::Vector{Term}           # Terms to sum over (in py terms[0])
    sum_i_neq_j_terms::Vector{Term}     # Terms to sum over (in py terms[1])
    non_sum_terms::Vector{Term}         # Terms that do not contain a sum (in py terms[2] - i=ind[0] + ... + i=ind[-1] + term_kappa+drive_terms)

    # Inner constructor with default values
    function DE_Term_Multi(exp_op::Term, which_ind::Vector{String}, sum_ind::String; sum_i_terms::Vector{Term}=Term[], sum_i_neq_j_terms::Vector{Term}=Term[], non_sum_terms::Vector{Term}=Term[])
        new(exp_op, which_ind, sum_ind, sum_i_terms, sum_i_neq_j_terms, non_sum_terms)
    end
end


function Base.show(io::IO, ::MIME"text/plain", x::DE_Term_Multi)
    latex_string = terms2str(x, do_latex=false)
    print(io, latex_string)
end
function Base.show(io::IO, ::MIME"text/latex", x::DE_Term_Multi)
    latex_string = terms2str(x, do_latex=true)
    print(io, latexstring(latex_string))
end

In [21]:
# Complete Liouvillian
function time_derivative_expectation_of_operator_multispin(exp_op::Term, index_delta::Bool=true, index_g::Bool=true)::DE_Term_Multi
    # Liouvillian = -i [H, \rho] \rho + \gamma D[\sigma_z] \rho + \Gamma D[\sigma_-] + \kappa D[a] \rho# get all indexes of exp_op
    which_ind, s, c = get_indexes(exp_op)  # s=has spin, c=has cavity
    # if i is in indexes change
    # in multi spin case, we construct the hamiltonian in decomposed way (see paper of this project)
    # so that the list all_terms has the structure:
    #               [ sum_i , sum_i \neq ind, i=ind[0] + ... + i=ind[-1] + term_kappa+drive_terms ]
    # discuss the different cases:
    # case 1 s + c: [       , sum_i \neq ind, i=ind[0] + ... + i=ind[-1] + term_kappa+drive_terms ]
    # case 2 s + _: [       ,               , i=ind[0] + ... + i=ind[-1]                          ]
    # case 3 _ + c: [ sum_i ,               ,                            + term_kappa+drive_terms ]
    which_case::Int = 4
    if s
        which_case -= 2
    end
    if c
        which_case -= 1
    end
    # Determine sum index
    sum_ind::String = "i" 
    i = "i"
    if i in which_ind
        for i in ["h", "g", "f", "e", "d", "c", "b", "a"]
            if !(i in which_ind)
                sum_ind = i
                break
            end
        end
    end
    de_term::DE_Term_Multi = DE_Term_Multi(exp_op, which_ind, sum_ind)

    delta_str, g_str = get_delta_and_g_str(sum_ind, index_delta, index_g)

    if which_case == 3          #### do_g_term (sum_i)
        a_sigma_plus = multiply_terms(make_term("-"), sigma_plus(sum_ind))  # a_- * sig_+
        a_dag_sigma_minus = multiply_terms(make_term("+"), sigma_minus(i))  # a_+ * sig_-
        sum_i = add_vars(commutator_terms(exp_op, add_terms(a_sigma_plus, a_dag_sigma_minus), -1.0im), g_str)
        de_term.sum_i_terms = combine_terms(sum_i)
    elseif which_case == 1      ####  do sum_i \neq ind term(s)
        de_term.sum_i_neq_j_terms = combine_terms(Trace_of_L_i_operator(exp_op, sum_ind, index_delta, index_g))
    end

    curr_terms::Vector{Term} = Term[]
    if which_case in [1, 2]  #### do terms with: i=ind[0] + ... + i=ind[-1] 
        for i in which_ind
            append!(curr_terms, Trace_of_L_i_operator(exp_op, i))
        end
    end
    # add spin independent terms
    brace_term = add_terms(make_term("+", 1.0im, raw"\sqrt{\kappa}\beta"), make_term("-", -1.0im, raw"\sqrt{\kappa}\beta^{*}"))
    drive_terms = commutator_terms(exp_op, brace_term, -1.0im)
    term_kappa = Trace_of_D_Operator(make_term("-"), exp_op, raw"\kappa")
    append!(curr_terms, add_terms(term_kappa, drive_terms))
    de_term.non_sum_terms = combine_terms(curr_terms)
    return de_term
end
# Test 
exp_op = make_term("-")
display(time_derivative_expectation_of_operator_multispin(exp_op))


d/dt⟨a⟩ = ∑ᵢ - g_i/2(i⟨xᵢ⟩ + ⟨yᵢ⟩) + √̅κβ - κ/2κ⟨a⟩